In [1]:
from pathlib import Path
import sys

import polars as pl

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

from ufc_scraper.utils import read_fighters


In [2]:
fighters = read_fighters().collect()

In [3]:
fighters.head()

fighter_id,fighter_link,first_name,last_name,nickname,height,weight,reach,stance,wins,losses,draws,belt,scraped_at,ingested_at
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""93fe7332d16c6ad9""","""http://www.ufcstats.com/fighte…","""Tom""","""Aaron""","""""","""--""","""155 lbs.""","""--""","""""","""5""","""3""","""0""","""""","""2026-09-05T14:33:24+00:00""","""2026-09-05T14:33:42+00:00"""
"""15df64c02b6b0fde""","""http://www.ufcstats.com/fighte…","""Danny""","""Abbadi""","""The Assassin""","""5' 11""""","""155 lbs.""","""--""","""Orthodox""","""4""","""6""","""0""","""""","""2026-09-05T14:33:24+00:00""","""2026-09-05T14:33:42+00:00"""
"""59a9d6dac61c2540""","""http://www.ufcstats.com/fighte…","""Nariman""","""Abbasov""","""Bayraktar""","""5' 8""""","""155 lbs.""","""66.0""""","""Orthodox""","""28""","""4""","""0""","""""","""2026-09-05T14:33:24+00:00""","""2026-09-05T14:33:42+00:00"""
"""4961467134abd8be""","""http://www.ufcstats.com/fighte…","""Darion""","""Abbey""","""""","""6' 2""""","""265 lbs.""","""80.0""""","""Orthodox""","""9""","""5""","""0""","""""","""2026-09-05T14:33:24+00:00""","""2026-09-05T14:33:42+00:00"""
"""b361180739bed4b0""","""http://www.ufcstats.com/fighte…","""David""","""Abbott""","""Tank""","""6' 0""""","""265 lbs.""","""--""","""Switch""","""10""","""15""","""0""","""""","""2026-09-05T14:33:24+00:00""","""2026-09-05T14:33:42+00:00"""


In [4]:
fighters.shape

(4616, 15)

In [5]:
fighters['belt'].unique()

belt
str
""""""


In [8]:
def standardize_dataframe(
    df: pl.DataFrame,
    empty_to_null: bool = False,
) -> pl.DataFrame:
    """
    Standardize a Polars DataFrame.

    - Strips whitespace from column names.
    - Strips whitespace from all string values.
    - Removes surrounding double quotes from string values.
    - Optionally converts empty strings to nulls.
    """

    # Clean column names
    df = df.rename(
        {
            col: col.strip()
            for col in df.columns
        }
    )

    string_cols = [
        col
        for col, dtype in df.schema.items()
        if dtype == pl.String
    ]

    exprs = []

    for col in string_cols:
        expr = (
            pl.col(col)
            .str.strip_chars()
            .str.strip_chars('"')
        )

        if empty_to_null:
            expr = (
                pl.when(expr == "")
                .then(None)
                .otherwise(expr)
            )

        exprs.append(expr.alias(col))

    return df.with_columns(*exprs)

In [ ]:
FIGHTER_COLS = [
    "fighter_id",
    "fighter_name",
    "nickname",
    "height_inches",
    "weight_lbs",
    "reach_inches",
    "stance",
    "wins",
    "losses",
    "draws",
    "scraped_at",
    "ingested_at",
]


def transform_fighters(
    fighters: pl.DataFrame,
) -> pl.DataFrame:

    return (
        fighters
        .with_columns(

            # Fighter name
            pl.concat_str(
                [
                    pl.col("first_name"),
                    pl.col("last_name"),
                ],
                separator=" ",
            )
            .str.strip_chars()
            .alias("fighter_name"),

            # Height: 5' 11" -> 71
            
            (
                pl.col("height")
                .str.extract(r"(\d+)'", 1)
                .cast(pl.Int64)
                * 12
                +
                pl.col("height")
                .str.extract(r"'\s*(\d+)", 1)
                .cast(pl.Float64)
            ).alias("height_inches"),

            # Weight: 155 lbs. -> 155
            pl.col("weight")
            .str.extract(r"(\d+)", 1)
            .cast(pl.Float64, strict=False)
            .alias("weight_lbs"),

            # Reach: 66.0" -> 66.0
            pl.col("reach")
            .str.extract(r"(\d+(?:\.\d+)?)", 1)
            .cast(pl.Float64, strict=False)
            .alias("reach_inches"),

            # Record columns
            pl.col("wins")
            .cast(pl.Int64, strict=False),

            pl.col("losses")
            .cast(pl.Int64, strict=False),

            pl.col("draws")
            .cast(pl.Int64, strict=False),

            # Belt flag
            pl.col("belt")
            .str.len_chars()
            .gt(0)
            .alias("belt"),

            # Clean text columns
            pl.col("fighter_id")
            .str.strip_chars(),

            pl.col("nickname")
            .str.strip_chars(),

            pl.col("stance")
            .str.strip_chars(),

            # Datetime columns
            pl.col("scraped_at")
            .str.to_datetime(
                format="%Y-%m-%dT%H:%M:%S%z"
            ),

            pl.col("ingested_at")
            .str.to_datetime(
                format="%Y-%m-%dT%H:%M:%S%z"
            ),
        )
        .select(FIGHTER_COLS)
    )

In [28]:
fighters_clean = (
    fighters
    .pipe(
        standardize_dataframe,
        empty_to_null=True,
    )
    .pipe(transform_fighters)
)

In [29]:
fighters_clean

fighter_id,fighter_name,nickname,height_inches,weight_lbs,reach_inches,stance,wins,losses,draws,scraped_at,ingested_at
str,str,str,f64,f64,f64,str,i64,i64,i64,"datetime[μs, UTC]","datetime[μs, UTC]"
"""93fe7332d16c6ad9""","""Tom Aaron""",null,null,155.0,null,null,5,3,0,2026-09-05 14:33:24 UTC,2026-09-05 14:33:42 UTC
"""15df64c02b6b0fde""","""Danny Abbadi""","""The Assassin""",71.0,155.0,null,"""Orthodox""",4,6,0,2026-09-05 14:33:24 UTC,2026-09-05 14:33:42 UTC
"""59a9d6dac61c2540""","""Nariman Abbasov""","""Bayraktar""",68.0,155.0,66.0,"""Orthodox""",28,4,0,2026-09-05 14:33:24 UTC,2026-09-05 14:33:42 UTC
"""4961467134abd8be""","""Darion Abbey""",null,74.0,265.0,80.0,"""Orthodox""",9,5,0,2026-09-05 14:33:24 UTC,2026-09-05 14:33:42 UTC
"""b361180739bed4b0""","""David Abbott""","""Tank""",72.0,265.0,null,"""Switch""",10,15,0,2026-09-05 14:33:24 UTC,2026-09-05 14:33:42 UTC
…,…,…,…,…,…,…,…,…,…,…,…
"""02d808afb96669de""","""Alex Zuniga""",null,null,145.0,null,null,6,3,0,2026-09-05 14:33:24 UTC,2026-09-05 14:33:42 UTC
"""1291dd6b8ab4d952""","""George Zuniga""",null,69.0,185.0,null,null,3,1,0,2026-09-05 14:33:24 UTC,2026-09-05 14:33:42 UTC
"""523af801b3429015""","""Allan Zuniga""","""Tigre""",67.0,155.0,70.0,"""Orthodox""",13,1,0,2026-09-05 14:33:24 UTC,2026-09-05 14:33:42 UTC


In [20]:
fighters_clean.filter(
    pl.col("height_inches").is_not_null()
)

fighter_id,fighter_name,nickname,height_inches,weight_lbs,reach_inches,stance,wins,losses,draws,scraped_at,ingested_at
str,str,str,i64,i64,f64,str,i64,i64,i64,"datetime[μs, UTC]","datetime[μs, UTC]"


In [17]:
fighters_clean.shape

(4616, 12)

In [21]:
fighters.head()

fighter_id,fighter_link,first_name,last_name,nickname,height,weight,reach,stance,wins,losses,draws,belt,scraped_at,ingested_at
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""93fe7332d16c6ad9""","""http://www.ufcstats.com/fighte…","""Tom""","""Aaron""","""""","""--""","""155 lbs.""","""--""","""""","""5""","""3""","""0""","""""","""2026-09-05T14:33:24+00:00""","""2026-09-05T14:33:42+00:00"""
"""15df64c02b6b0fde""","""http://www.ufcstats.com/fighte…","""Danny""","""Abbadi""","""The Assassin""","""5' 11""""","""155 lbs.""","""--""","""Orthodox""","""4""","""6""","""0""","""""","""2026-09-05T14:33:24+00:00""","""2026-09-05T14:33:42+00:00"""
"""59a9d6dac61c2540""","""http://www.ufcstats.com/fighte…","""Nariman""","""Abbasov""","""Bayraktar""","""5' 8""""","""155 lbs.""","""66.0""""","""Orthodox""","""28""","""4""","""0""","""""","""2026-09-05T14:33:24+00:00""","""2026-09-05T14:33:42+00:00"""
"""4961467134abd8be""","""http://www.ufcstats.com/fighte…","""Darion""","""Abbey""","""""","""6' 2""""","""265 lbs.""","""80.0""""","""Orthodox""","""9""","""5""","""0""","""""","""2026-09-05T14:33:24+00:00""","""2026-09-05T14:33:42+00:00"""
"""b361180739bed4b0""","""http://www.ufcstats.com/fighte…","""David""","""Abbott""","""Tank""","""6' 0""""","""265 lbs.""","""--""","""Switch""","""10""","""15""","""0""","""""","""2026-09-05T14:33:24+00:00""","""2026-09-05T14:33:42+00:00"""


In [22]:
fighters.filter(
    pl.col("height") != "--")

fighter_id,fighter_link,first_name,last_name,nickname,height,weight,reach,stance,wins,losses,draws,belt,scraped_at,ingested_at
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""15df64c02b6b0fde""","""http://www.ufcstats.com/fighte…","""Danny""","""Abbadi""","""The Assassin""","""5' 11""""","""155 lbs.""","""--""","""Orthodox""","""4""","""6""","""0""","""""","""2026-09-05T14:33:24+00:00""","""2026-09-05T14:33:42+00:00"""
"""59a9d6dac61c2540""","""http://www.ufcstats.com/fighte…","""Nariman""","""Abbasov""","""Bayraktar""","""5' 8""""","""155 lbs.""","""66.0""""","""Orthodox""","""28""","""4""","""0""","""""","""2026-09-05T14:33:24+00:00""","""2026-09-05T14:33:42+00:00"""
"""4961467134abd8be""","""http://www.ufcstats.com/fighte…","""Darion""","""Abbey""","""""","""6' 2""""","""265 lbs.""","""80.0""""","""Orthodox""","""9""","""5""","""0""","""""","""2026-09-05T14:33:24+00:00""","""2026-09-05T14:33:42+00:00"""
"""b361180739bed4b0""","""http://www.ufcstats.com/fighte…","""David""","""Abbott""","""Tank""","""6' 0""""","""265 lbs.""","""--""","""Switch""","""10""","""15""","""0""","""""","""2026-09-05T14:33:24+00:00""","""2026-09-05T14:33:42+00:00"""
"""3329d692aea4dc28""","""http://www.ufcstats.com/fighte…","""Hamdy""","""Abdelwahab""","""The Hammer""","""6' 2""""","""265 lbs.""","""72.0""""","""Southpaw""","""7""","""1""","""0""","""""","""2026-09-05T14:33:24+00:00""","""2026-09-05T14:33:42+00:00"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""108afe61a26bcbf4""","""http://www.ufcstats.com/fighte…","""Igor""","""Zinoviev""","""Houdini""","""6' 1""""","""199 lbs.""","""--""","""Orthodox""","""4""","""1""","""2""","""""","""2026-09-05T14:33:24+00:00""","""2026-09-05T14:33:42+00:00"""
"""1291dd6b8ab4d952""","""http://www.ufcstats.com/fighte…","""George""","""Zuniga""","""""","""5' 9""""","""185 lbs.""","""--""","""""","""3""","""1""","""0""","""""","""2026-09-05T14:33:24+00:00""","""2026-09-05T14:33:42+00:00"""
"""523af801b3429015""","""http://www.ufcstats.com/fighte…","""Allan""","""Zuniga""","""Tigre""","""5' 7""""","""155 lbs.""","""70.0""""","""Orthodox""","""13""","""1""","""0""","""""","""2026-09-05T14:33:24+00:00""","""2026-09-05T14:33:42+00:00"""


In [23]:
fighters.select(
    pl.col("height"),
    pl.col("height")
        .str.extract(r"(\d+)'", 1)
        .alias("feet"),
    pl.col("height")
        .str.extract(r"'\s*(\d+)", 1)
        .alias("inches"),
)

height,feet,inches
str,str,str
"""--""",null,null
"""5' 11""""","""5""","""11"""
"""5' 8""""","""5""","""8"""
"""6' 2""""","""6""","""2"""
"""6' 0""""","""6""","""0"""
…,…,…
"""--""",null,null
"""5' 9""""","""5""","""9"""
"""5' 7""""","""5""","""7"""
